# Outlier Detection and Preprocessing

This notebook applies the first fitted data-preparation steps for modeling.

Important rule: preprocessing is fit on the training split only, then applied to validation and test sets. This avoids data leakage.

## 1. Environment Setup

Load libraries, connect to the project source code, and define the random seed.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_root = Path.cwd().resolve().parents[0]
src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from credit_risk_platform.data.german_credit import interim_german_credit_dir
from credit_risk_platform.utils.io import load_csv

RANDOM_STATE = 42

print("project_root:", project_root)
print("src_dir exists:", src_dir.exists())

## 2. Load the Standardized Dataset

Load the analysis-ready dataset created from the open German Credit source files.

In [ ]:
interim_dir = interim_german_credit_dir(project_root)
df = load_csv(interim_dir / "german_credit_standardized.csv")

print("Shape:", df.shape)
df.head()

## 3. Define Features and Target

Exclude identifier and target-derived columns from the modeling feature set.

In [ ]:
target_col = "TARGET"
identifier_cols = ["applicant_id"]
target_label_cols = ["risk_class", "risk_label"]
exclude_from_features = identifier_cols + target_label_cols + [target_col]

feature_cols = [col for col in df.columns if col not in exclude_from_features]

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Feature count:", len(feature_cols))
display(pd.DataFrame({"feature_cols": pd.Series(feature_cols)}))

## 4. Recreate Train, Validation, and Test Splits

Use the same stratified split strategy from Notebook 05 so preprocessing is fit only on training data.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("Test:", X_test.shape)

## 5. Identify Numeric and Categorical Features

Numeric and categorical variables require different preprocessing treatment.

In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

display(pd.DataFrame({"numeric_features": pd.Series(numeric_features)}))
display(pd.DataFrame({"categorical_features": pd.Series(categorical_features)}))

## 6. Review Training-Only Outliers

Outlier thresholds are reviewed using training data only. Validation and test data should not influence fitted preprocessing decisions.

In [ ]:
outlier_rows = []

for col in numeric_features:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_mask = (X_train[col] < lower_bound) | (X_train[col] > upper_bound)

    outlier_rows.append(
        {
            "feature": col,
            "lower_bound": round(lower_bound, 2),
            "upper_bound": round(upper_bound, 2),
            "outlier_count": int(outlier_mask.sum()),
            "outlier_pct": round(outlier_mask.mean() * 100, 2),
        }
    )

train_outlier_summary = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
train_outlier_summary

## 7. Define Preprocessing Strategy

For the baseline modeling stage, use a simple, transparent preprocessing pipeline:
- numeric features: median imputation and standard scaling
- categorical features: most-frequent imputation and one-hot encoding

Outlier clipping is not fitted yet. The first baseline should show how the model behaves with a standard pipeline before adding extra transformations.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)

preprocessor

## 8. Fit Preprocessor on Training Data Only

The preprocessor learns imputation values, scaling statistics, and encoding categories from `X_train` only.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)
print("Processed test shape:", X_test_processed.shape)

## 9. Recover Processed Feature Names

Feature names are needed for model interpretation, coefficient review, and later explainability.

In [ ]:
processed_feature_names = preprocessor.get_feature_names_out()

processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index,
)

processed_train_df.head()

## 10. Save Preprocessing Summary Artifact

Record the preprocessing design so it is clear what was fitted and where leakage was controlled.

In [ ]:
artifacts_dir = project_root / "artifacts" / "profiles"
artifacts_dir.mkdir(parents=True, exist_ok=True)

preprocessing_summary = {
    "random_state": RANDOM_STATE,
    "target_col": target_col,
    "fit_policy": "fit preprocessing on training data only",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "numeric_steps": ["median_imputation", "standard_scaling"],
    "categorical_steps": ["most_frequent_imputation", "one_hot_encoding_handle_unknown_ignore"],
    "outlier_policy": "reviewed on training data; no clipping in baseline preprocessing",
    "processed_shape": {
        "train": list(X_train_processed.shape),
        "validation": list(X_valid_processed.shape),
        "test": list(X_test_processed.shape),
    },
    "processed_feature_count": int(len(processed_feature_names)),
}

preprocessing_summary_path = artifacts_dir / "preprocessing_summary.json"
with preprocessing_summary_path.open("w") as f:
    json.dump(preprocessing_summary, f, indent=2)

preprocessing_summary_path

## 11. Preprocessing Decision Summary

The baseline preprocessing pipeline is now defined and fitted on the training set only. Numeric features use median imputation and scaling. Categorical features use most-frequent imputation and one-hot encoding.

Outliers are retained for the first baseline model because `credit_amount` and `duration_months` likely contain genuine lending risk signals. More advanced transformations can be compared later if baseline model diagnostics suggest they are needed.

Next notebook: `07_baseline_models.ipynb`.